# Kick, Repair, Squeeze: Basin Hopping for Square Packings

## experiment 02 · is there a tighter packing next door?

A packing can be locally tight and still not the best one nearby. This experiment starts from the squeezed packing and hops:
it **kicks** the packing (shrinks it slightly, or jiggles one local patch of squares), **repairs** the overlaps with the contact LP
in penalty mode, **squeezes** again and keeps the result only if it is valid and shorter. Mid-size puzzles (`10 ≤ n ≤ 400`)
that already beat the grid get a fixed time each.

| stage | what it does |
|---|---|
| metric | official metric, verbatim, SHA-256 checked |
| baseline | certified submission and per-puzzle scores of our merge notebook |
| contact LP | trust-region LP; with a penalty weight it also removes overlaps |
| kick and repair | shrink or jiggle → repair → squeeze, best valid packing kept |
| write | merged `submission.csv`, scored end to end by the metric, and `improved.csv` |

### Credits

| author | notebook | used for |
|---|---|---|
| Kaggle (hosts) | [metric/squares-in-squares](https://www.kaggle.com/code/metric/squares-in-squares) | the metric, verbatim (Apache-2.0) |
| alex chilton | [Fault-line DP + Basin Hopping](https://www.kaggle.com/code/alexchilton/fault-line-dp-basin-hopping-lb-22-2620) | basin hopping on packings |
| Malyshev Danil | [Neighbour Transfer + Squeeze](https://www.kaggle.com/code/malyshevdanil/squares-in-squares-neighbour-transfer-squeeze) | shrink-and-repair |
| sigmaborov | [Certified Contact Polish](https://www.kaggle.com/code/sigmaborov/squares-in-squares-certified-contact-polish) | the contact-LP polish idea |
| Nilanjan Saha | [Certified Merge + Projection Quench](https://www.kaggle.com/code/nilanjansaha123/certified-merge-projection-quench-gpu-screen) | quench after a perturbation |
| everyone in the merge | see [Every Public Packing, Certified](https://www.kaggle.com/code/flexonafft/every-public-packing-certified) | the starting packings |

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────────────────
CFG = dict(
    WORKERS=4,                 # Kaggle CPU cores
    BUDGET_S=3.0 * 3600,       # wall clock for the hops; the final metric pass (~10 min) comes after
    PER_PUZZLE_S=150,          # every target gets this long
    N_RANGE=(10, 400),         # puzzles that beat the grid inside this range
    R0=1e-3,                   # trust region of the LP
    ITERS=120,                 # LP steps per squeeze / repair
    MARGIN=2e-12,
    TOL=2e-14,
    MIN_GAIN=1e-12,
    REPAIR_W=10.0,             # penalty on overlap depth while repairing
    P_SHRINK=0.4,              # share of kicks that shrink the whole packing (the rest jiggle a patch)
    SHRINK_LOG10=(-6.0, -3.5), # shrink factor 10^U(a, b)
    PATCH_RADIUS=(1.5, 3.5),   # jiggled patch radius, in square sides
    JIGGLE_LOG10=(-4.0, -2.0), # jiggle std 10^U(a, b) for positions and angles
    SEED=2026,
    RESCUE_DELTAS=[4e-16, 2e-15, 1e-14, 1e-13, 1e-12],
)

In [ ]:
# ── Session: paths, logging and house style ───────────────────────────────────────────────────
import os, sys, csv, json, math, time, hashlib, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

NB_T0 = time.time()
warnings.filterwarnings("ignore")
INPUT = Path("/kaggle/input")
ROOT = Path("/kaggle/working"); ROOT.mkdir(parents=True, exist_ok=True); os.chdir(ROOT)
COLS = ["n", "square_id", "x", "y", "angle"]
CEIL = {n: math.ceil(math.sqrt(n)) / math.sqrt(n) - 1 for n in range(1, 1001)}   # grid score per puzzle


def log(msg: str) -> None:
    """Print one line prefixed with the elapsed notebook time."""
    print(f"[{time.time() - NB_T0:7.0f}s] {msg}", flush=True)


# House style (as in our other competition notebooks): one accent colour means "this is the point".
INK, MUTED, GRID = "#1f2328", "#8b949e", "#e6e8eb"
ACCENT, SECOND = "#d9480f", "#1c7ed6"
PALETTE = [ACCENT, SECOND, "#495057", "#adb5bd", "#2b8a3e", "#862e9c"]


def apply() -> None:
    """Set matplotlib defaults. Safe to call more than once."""
    mpl.rcParams.update({
        "figure.figsize": (9, 4.6), "figure.dpi": 110, "savefig.dpi": 160,
        "figure.facecolor": "white", "axes.facecolor": "white",
        "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "semibold", "axes.titlelocation": "left",
        "axes.labelsize": 11, "axes.labelcolor": INK, "text.color": INK,
        "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 10, "ytick.labelsize": 10,
        "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
        "legend.frameon": False, "axes.prop_cycle": mpl.cycler(color=PALETTE),
    })


apply()
log(f"inputs mounted: {sorted(p.name for p in INPUT.iterdir()) if INPUT.exists() else 'none'}")

## The metric, verbatim

The hosts' metric ([metric/squares-in-squares](https://www.kaggle.com/code/metric/squares-in-squares)) is written to `sis_metric.py`
unchanged and checked by SHA-256. Every packing this notebook keeps is accepted by this code, on the exact doubles Kaggle will read.

In [ ]:
%%writefile sis_metric.py
# Verbatim copy of the competition metric, Kaggle notebook metric/squares-in-squares (pulled 2026-11-02). Do not edit.
"""Metric for the "Packing N Unit Squares into the Smallest Square" competition.

Participants submit packings of n unit squares (n=1..1000) inside a bounding
square.  Each square is specified by its center (x, y) and rotation angle.
Coordinates must be plain decimal literals, optionally in scientific notation
(e.g. ``2.70710678118654752440`` or ``1.5e-3``).  Symbolic / closed-form
expressions are not accepted -- submit their decimal expansion instead, at as
many digits as you like.

Scoring
-------
For each puzzle n the bounding side length L is measured
translation-invariantly as ``max(max_x - min_x, max_y - min_y)`` over all
corners.  The per-puzzle score is the excess over the area lower bound

    score(n) = L / sqrt(n) - 1

which is 0.0 exactly when the packing achieves the area lower bound (only
possible for perfect squares packed on the grid) and grows above 0 as the
bounding square wastes area.  **Lower is better**; total score is the sum
across n=1..N.

+-------------------+---------------------------------------------------------+
| Condition         | Score                                                   |
+===================+=========================================================+
| feasible packing  | L / sqrt(n) - 1     (>= 0 always)                       |
+-------------------+---------------------------------------------------------+
| infeasible/invalid| L_0 / sqrt(n) - 1   (grid ceiling — no worse than       |
|                   |                      submitting the trivial grid)       |
+-------------------+---------------------------------------------------------+

For reference: ``L_B = sqrt(n)`` is the area lower bound and
``L_0 = ceil(sqrt(n))`` is the trivial grid packing.  The best-known side
length plays no part -- beating a record just yields a smaller excess -- so
this metric reads no baseline values at all.

Feasibility is checked with interval arithmetic (mpmath) at escalating
precision to certify that no two squares overlap.

Kaggle entry point
------------------
``score(solution, submission, row_id_column_name)`` — called by the Kaggle
backend.  ``solution`` is a DataFrame loaded from ``solution.csv``; only its
``n`` column is read, to fix which puzzles are scored (and which absent ones
are charged the grid ceiling).  ``submission`` is the participant CSV with
columns ``n, square_id, x, y, angle``.

As a side effect it writes ``submission_details.csv``, one row per puzzle with
columns ``n`` and ``L``::

    n,L
    1,1.00000000000000000000000000000000000000000000000000
    2,2.00000000000000000000000000000000000000000000000000
    ...

``L`` is the measured bounding side for puzzle n -- not the per-puzzle score --
written with exactly 50 digits after the decimal point.  Those digits are
significant: L is measured in ``L_DPS``-digit mpmath arithmetic straight from
the submitted decimal strings, not from a double.  Puzzles with no accepted L
(absent from the submission, unparseable, failing column validation, or
rejected as infeasible) are written as zero.
"""

import contextlib
import csv
import io
import math
import multiprocessing
import re
from collections import defaultdict
from collections.abc import Callable, Sequence
from dataclasses import dataclass, field
from fractions import Fraction
from pathlib import Path

import mpmath
import pandas as pd

# ---------------------------------------------------------------------------
# interval
# ---------------------------------------------------------------------------

IVal = mpmath.ctx_iv.ivmpf

# Working precision for the (non-interval) measurement of the bounding side L,
# and the number of digits after the decimal point written for it in
# submission_details.csv.  The margin between the two keeps every written digit
# correct: the only rounding is in the trig of the corner computation, tens of
# digits below the last digit reported.
L_DPS = 60
L_DECIMALS = 50


@contextlib.contextmanager
def working_dps(dps: int):
    """Run a block with both mpmath contexts at ``dps`` digits, then restore them.

    ``mpmath.iv`` carries its own precision, independent of ``mpmath.mp``:
    setting only ``mpmath.mp.dps`` leaves every interval operation at the
    default 53 bits, which caps what a contact can be certified to at ~1e-13 no
    matter how far the escalation sequence climbs.  Both contexts are set here
    because the interval work is interleaved with plain ``mp`` arithmetic.

    >>> with working_dps(500):
    ...     width = mpmath.mpf(mpmath.iv.mpf('0.1').delta)
    >>> width < mpmath.mpf(10) ** -500
    True
    """
    old_iv_dps = mpmath.iv.dps
    old_mp_dps = mpmath.mp.dps
    try:
        mpmath.iv.dps = dps
        mpmath.mp.dps = dps
        yield
    finally:
        mpmath.iv.dps = old_iv_dps
        mpmath.mp.dps = old_mp_dps


def make_interval(value: str, dps: int = 50) -> IVal:
    with working_dps(dps):
        return mpmath.iv.mpf(value)


def iv_cos(x: IVal) -> IVal:
    return mpmath.iv.cos(x)


def iv_sin(x: IVal) -> IVal:
    return mpmath.iv.sin(x)


_ESCALATION_SEQUENCE = [50, 100, 200, 500]


def escalate(
    fn: Callable[[int], bool | None],
    max_dps: int = 500,
    start_dps: int = 50,
) -> bool | None:
    for dps in _ESCALATION_SEQUENCE:
        if dps < start_dps:
            continue
        if dps > max_dps:
            break
        result = fn(dps)
        if result is not None:
            return result
    return None


def iv_le(a: IVal, b: IVal) -> bool | None:
    """Certified less-or-equal: True if a <= b, False if a > b, None if undecided.

    Comparing two overlapping intervals is indeterminate.  mpmath 1.3 reports
    that as None, 1.4 raises ValueError; both mean "undecided" here.
    """
    try:
        return a <= b
    except ValueError:
        return None


def iv_min(values: Sequence[IVal]) -> IVal:
    """Interval extension of ``min``: the tightest interval containing the true minimum.

    Comparison operators are useless for this -- two overlapping intervals have
    no determinate order -- so take it endpoint-wise instead.  min of the lower
    bounds is a lower bound on the true min, and min of the upper bounds is an
    upper bound on it, whichever element turns out to be smallest.
    """
    return mpmath.iv.mpf([min(v.a for v in values), min(v.b for v in values)])


def iv_max(values: Sequence[IVal]) -> IVal:
    """Interval extension of ``max``. See :func:`iv_min`."""
    return mpmath.iv.mpf([max(v.a for v in values), max(v.b for v in values)])


# ---------------------------------------------------------------------------
# parser
# ---------------------------------------------------------------------------

# Bounds the cost of parsing one coordinate.  Far above any useful precision
# (L is reported to 50 decimals), so no honest submission comes near it.
_MAX_LITERAL_CHARS = 100

# Coordinate, angle, and sweep bounds.  For n <= 1000 unit squares, valid
# packings have side L <= 32.  Bounding magnitudes prevents float64/mpmath
# discrepancies and precision loss under extreme translation or rotation.
_MAX_COORD = 1e4
_MAX_ANGLE = 100.0
_AABB_PAD = 1e-7

_DECIMAL_RE = re.compile(r'^[+-]?(\d+\.?\d*|\.\d+)([eE][+-]?\d+)?$')


def _is_decimal(s: str) -> bool:
    return len(s) <= _MAX_LITERAL_CHARS and bool(_DECIMAL_RE.match(s.strip()))


def _check_decimal(s: str) -> str:
    """Return the stripped literal, or raise if it is not a plain decimal.

    >>> _check_decimal(" 2.70710678118654752440 ")
    '2.70710678118654752440'
    >>> _check_decimal("1.5e-3")
    '1.5e-3'
    >>> _check_decimal("1 + sqrt(2)")
    Traceback (most recent call last):
        ...
    ValueError: not a decimal number: '1 + sqrt(2)'
    """
    s = s.strip()
    if not _is_decimal(s):
        shown = s if len(s) <= 40 else s[:40] + '...'
        raise ValueError(f'not a decimal number: {shown!r}')
    return s


def parse_value(s: str, dps: int = 50) -> IVal:
    """Parse a decimal literal into an enclosing interval at ``dps`` digits."""
    return make_interval(_check_decimal(s), dps)


def parse_value_mp(s: str, dps: int = L_DPS) -> mpmath.mpf:
    """Parse a decimal literal into an ``mpf`` at ``dps`` digits.

    >>> mpmath.nstr(parse_value_mp("2.70710678118654752440"), 21, strip_zeros=False)
    '2.70710678118654752440'
    """
    s = _check_decimal(s)
    old_dps = mpmath.mp.dps
    try:
        mpmath.mp.dps = dps
        return mpmath.mpf(s)
    finally:
        mpmath.mp.dps = old_dps


def parse_value_float(s: str) -> float:
    """Parse a decimal literal to float.

    Values too large for a double are rejected rather than becoming ``inf``,
    which would poison every downstream geometry check:

    >>> parse_value_float("3.5")
    3.5
    >>> parse_value_float("-2.5")
    -2.5
    >>> parse_value_float("1.5e-3")
    0.0015
    >>> parse_value_float("1e400")
    Traceback (most recent call last):
        ...
    ValueError: value out of range for a double: '1e400'
    """
    s = _check_decimal(s)
    value = float(s)
    if not math.isfinite(value):
        raise ValueError(f'value out of range for a double: {s!r}')
    return value


# ---------------------------------------------------------------------------
# scoring
# ---------------------------------------------------------------------------


def score_puzzle(n: int, s_certified: float) -> float:
    """Score a single puzzle: L / sqrt(n) - 1 (lower is better).

    Result is >= 0 for any feasible packing (equality only for perfect-square
    grid packings).  The best-known side length s* plays no part: beating a
    record simply yields a smaller excess.

    >>> score_puzzle(4, 2.0)
    0.0
    >>> round(score_puzzle(5, 2.707), 6)
    0.210607
    >>> score_puzzle(4, 4.0)
    1.0
    """
    L_B = math.sqrt(n)
    return s_certified / L_B - 1.0


def infeasible_score(n: int) -> float:
    """Penalty score for an infeasible / invalid submission for puzzle n.

    Equal to submitting the trivial grid packing: L_0 / sqrt(n) - 1.
    """
    return math.ceil(math.sqrt(n)) / math.sqrt(n) - 1.0


# ---------------------------------------------------------------------------
# geometry
# ---------------------------------------------------------------------------


@dataclass
class Square:
    x: float
    y: float
    angle: float
    x_str: str = ''
    y_str: str = ''
    angle_str: str = ''
    _corners_float: list[tuple[float, float]] = field(default_factory=list, repr=False)

    def corners_float(self) -> list[tuple[float, float]]:
        if not self._corners_float:
            self._corners_float = compute_corners_float(self.x, self.y, self.angle)
        return self._corners_float


def compute_corners_float(cx: float, cy: float, angle: float) -> list[tuple[float, float]]:
    """Return the four corners of a unit square centered at (cx, cy) rotated by angle.

    >>> corners = compute_corners_float(0.5, 0.5, 0.0)
    >>> [(round(x, 1), round(y, 1)) for x, y in corners]
    [(1.0, 1.0), (0.0, 1.0), (0.0, 0.0), (1.0, 0.0)]
    """
    c = math.cos(angle)
    s = math.sin(angle)
    h = 0.5
    dx1, dy1 = h * c - h * s, h * s + h * c
    dx2, dy2 = -h * c - h * s, -h * s + h * c
    return [
        (cx + dx1, cy + dy1),
        (cx + dx2, cy + dy2),
        (cx - dx1, cy - dy1),
        (cx - dx2, cy - dy2),
    ]


def compute_corners_mp(cx: mpmath.mpf, cy: mpmath.mpf, angle: mpmath.mpf) -> list[tuple[mpmath.mpf, mpmath.mpf]]:
    """``compute_corners_float`` at the ambient ``mpmath.mp`` precision."""
    c = mpmath.cos(angle)
    s = mpmath.sin(angle)
    h = mpmath.mpf(1) / 2
    dx1, dy1 = h * c - h * s, h * s + h * c
    dx2, dy2 = -h * c - h * s, -h * s + h * c
    return [
        (cx + dx1, cy + dy1),
        (cx + dx2, cy + dy2),
        (cx - dx1, cy - dy1),
        (cx - dx2, cy - dy2),
    ]


def measure_side_mp(squares: list['Square'], dps: int = L_DPS) -> mpmath.mpf:
    """Bounding side ``max(max_x - min_x, max_y - min_y)`` at ``dps`` digits.

    Measured from the submitted decimal strings translated relative to Square 0,
    so coordinates carry high precision without loss due to global translation.

    >>> unit = Square(0.5, 0.5, 0.0, x_str="0.5", y_str="0.5", angle_str="0")
    >>> measure_side_mp([unit])
    mpf('1.0')
    """
    old_dps = mpmath.mp.dps
    try:
        mpmath.mp.dps = dps
        xs: list[mpmath.mpf] = []
        ys: list[mpmath.mpf] = []
        x0 = parse_value_mp(squares[0].x_str, dps) if squares else mpmath.mpf(0)
        y0 = parse_value_mp(squares[0].y_str, dps) if squares else mpmath.mpf(0)
        for sq in squares:
            corners = compute_corners_mp(
                parse_value_mp(sq.x_str, dps) - x0,
                parse_value_mp(sq.y_str, dps) - y0,
                parse_value_mp(sq.angle_str, dps),
            )
            for x, y in corners:
                xs.append(x)
                ys.append(y)
        return max(max(xs) - min(xs), max(ys) - min(ys))
    finally:
        mpmath.mp.dps = old_dps


def compute_corners_interval(cx_iv: IVal, cy_iv: IVal, angle_iv: IVal) -> list[tuple[IVal, IVal]]:
    c = iv_cos(angle_iv)
    s = iv_sin(angle_iv)
    h = mpmath.iv.mpf('0.5')
    dx1, dy1 = h * c - h * s, h * s + h * c
    dx2, dy2 = -h * c - h * s, -h * s + h * c
    return [
        (cx_iv + dx1, cy_iv + dy1),
        (cx_iv + dx2, cy_iv + dy2),
        (cx_iv - dx1, cy_iv - dy1),
        (cx_iv - dx2, cy_iv - dy2),
    ]


def compute_aabb(
    corners: list[tuple[float, float]],
    pad: float = _AABB_PAD,
) -> tuple[float, float, float, float]:
    xs = [c[0] for c in corners]
    ys = [c[1] for c in corners]
    return (min(xs) - pad, min(ys) - pad, max(xs) + pad, max(ys) + pad)


def aabb_overlap(
    a: tuple[float, float, float, float],
    b: tuple[float, float, float, float],
) -> bool:
    return not (a[2] < b[0] or b[2] < a[0] or a[3] < b[1] or b[3] < a[1])


def _project_corners_float(corners: list[tuple[float, float]], ax: float, ay: float) -> tuple[float, float]:
    projs = [ax * cx + ay * cy for cx, cy in corners]
    return (min(projs), max(projs))


def sat_check_float(sq1: Square, sq2: Square, undecided_tol: float = 1e-8) -> str:
    """Check overlap between two unit squares using the Separating Axis Theorem.

    Returns ``"separated"``, ``"overlapping"``, or ``"undecided"``.

    >>> sat_check_float(Square(0.5, 0.5, 0.0), Square(5.0, 5.0, 0.0))
    'separated'
    >>> sat_check_float(Square(0.5, 0.5, 0.0), Square(0.6, 0.6, 0.0))
    'overlapping'
    """
    c1 = sq1.corners_float()
    c2 = sq2.corners_float()

    axes = _get_axes_float(sq1.angle, sq2.angle)

    min_overlap = float('inf')
    for ax, ay in axes:
        min1, max1 = _project_corners_float(c1, ax, ay)
        min2, max2 = _project_corners_float(c2, ax, ay)
        overlap = min(max1, max2) - max(min1, min2)
        if overlap < -undecided_tol:
            return 'separated'
        min_overlap = min(min_overlap, overlap)

    if min_overlap > undecided_tol:
        return 'overlapping'
    return 'undecided'


def _get_axes_float(angle1: float, angle2: float) -> list[tuple[float, float]]:
    c1, s1 = math.cos(angle1), math.sin(angle1)
    c2, s2 = math.cos(angle2), math.sin(angle2)
    return [(c1, s1), (-s1, c1), (c2, s2), (-s2, c2)]


def sat_check_interval(
    sq1_corners: list[tuple[IVal, IVal]],
    sq2_corners: list[tuple[IVal, IVal]],
    angle1_iv: IVal,
    angle2_iv: IVal,
) -> bool | None:
    c1_iv, s1_iv = iv_cos(angle1_iv), iv_sin(angle1_iv)
    c2_iv, s2_iv = iv_cos(angle2_iv), iv_sin(angle2_iv)
    axes = [(c1_iv, s1_iv), (-s1_iv, c1_iv), (c2_iv, s2_iv), (-s2_iv, c2_iv)]

    has_undecided = False
    for ax, ay in axes:
        projs1 = [ax * cx + ay * cy for cx, cy in sq1_corners]
        projs2 = [ax * cx + ay * cy for cx, cy in sq2_corners]

        # Endpoint-wise, via iv_min/iv_max: ordering two overlapping intervals
        # with < is indeterminate, and skipping the projections that compare
        # indeterminately would narrow the span -- which shrinks the overlap and
        # biases the test towards wrongly certifying a pair as separated.
        min1, max1 = iv_min(projs1), iv_max(projs1)
        min2, max2 = iv_min(projs2), iv_max(projs2)

        overlap = iv_min([max1, max2]) - iv_max([min1, min2])
        le_result = iv_le(overlap, mpmath.iv.mpf('0'))
        if le_result is True:
            return True
        if le_result is None:
            has_undecided = True

    if has_undecided:
        return None
    return False


def sort_and_sweep(squares: list[Square]) -> list[tuple[int, int]]:
    n = len(squares)
    if n <= 1:
        return []

    aabbs = []
    for i, sq in enumerate(squares):
        corners = sq.corners_float()
        aabb = compute_aabb(corners)
        aabbs.append((i, aabb))

    aabbs.sort(key=lambda x: x[1][0])

    pairs = []
    for i_idx in range(n):
        i, aabb_i = aabbs[i_idx]
        for j_idx in range(i_idx + 1, n):
            j, aabb_j = aabbs[j_idx]
            if aabb_j[0] > aabb_i[2]:
                break
            if aabb_overlap(aabb_i, aabb_j):
                pairs.append((i, j) if i < j else (j, i))

    return pairs


def verify_puzzle(
    squares: list[Square],
) -> tuple[bool, str]:
    """Check that no two squares overlap, using float SAT then interval arithmetic.

    >>> s1 = Square(0.5, 0.5, 0.0, x_str="0.5", y_str="0.5", angle_str="0")
    >>> s2 = Square(1.5, 0.5, 0.0, x_str="1.5", y_str="0.5", angle_str="0")
    >>> verify_puzzle([s1, s2])
    (True, 'feasible')
    >>> verify_puzzle([Square(0.5, 0.5, 0.0), Square(0.6, 0.6, 0.0)])
    (False, 'Squares 0 and 1 overlap (float check)')
    """
    pairs = sort_and_sweep(squares)
    undecided_pairs = []
    for i, j in pairs:
        result = sat_check_float(squares[i], squares[j])
        if result == 'overlapping':
            return (False, f'Squares {i} and {j} overlap (float check)')
        if result == 'undecided':
            undecided_pairs.append((i, j))

    if undecided_pairs:

        def check_overlap_certified(dps: int) -> bool | None:
            # The whole pair loop runs inside the precision context: parsing at dps
            # digits is not enough, since every later interval operation widens its
            # result to whatever precision mpmath.iv is set to at the time.
            with working_dps(dps):
                still_undecided = []
                for i, j in undecided_pairs:
                    sq_i, sq_j = squares[i], squares[j]
                    cx_i = parse_value(sq_i.x_str, dps)
                    cy_i = parse_value(sq_i.y_str, dps)
                    ai = parse_value(sq_i.angle_str, dps)
                    cx_j = parse_value(sq_j.x_str, dps)
                    cy_j = parse_value(sq_j.y_str, dps)
                    aj = parse_value(sq_j.angle_str, dps)
                    corners_i = compute_corners_interval(cx_i, cy_i, ai)
                    corners_j = compute_corners_interval(cx_j, cy_j, aj)
                    result = sat_check_interval(corners_i, corners_j, ai, aj)
                    if result is False:
                        return False
                    if result is None:
                        still_undecided.append((i, j))
                if still_undecided:
                    return None
                return True

        result = escalate(check_overlap_certified)
        if result is False:
            return (False, 'Squares overlap (certified)')
        if result is None:
            return (False, 'Overlap undecidable at max precision')

    return (True, 'feasible')


# ---------------------------------------------------------------------------
# main / Kaggle entry point
# ---------------------------------------------------------------------------


class ParticipantVisibleError(Exception):
    pass


MAX_N = 1000
KAGGLE_CORES = 4
MIN_PUZZLES_FOR_MP = 4


def _parse_submission_csv(text: str) -> dict[int, list[dict]]:
    reader = csv.DictReader(io.StringIO(text))
    required_cols = {'n', 'square_id', 'x', 'y', 'angle'}
    if not required_cols.issubset(set(reader.fieldnames or [])):
        missing = required_cols - set(reader.fieldnames or [])
        raise ParticipantVisibleError(f'Missing columns: {missing}')

    puzzles: dict[int, list[dict]] = defaultdict(list)
    for row_num, row in enumerate(reader, start=2):
        try:
            n = int(row['n'])
        except (ValueError, TypeError):
            raise ParticipantVisibleError(f'Row {row_num}: invalid n value: {row["n"]!r}')
        if n < 1 or n > MAX_N:
            raise ParticipantVisibleError(f'Row {row_num}: n={n} out of range [1, {MAX_N}]')
        puzzles[n].append(row)

    return dict(puzzles)


def _validate_puzzle(n: int, rows: list[dict]) -> str | None:
    if len(rows) != n:
        return f'n={n}: expected {n} rows, got {len(rows)}'

    square_ids = set()
    for row in rows:
        try:
            sid = int(row['square_id'])
        except (ValueError, TypeError):
            return f'n={n}: invalid square_id: {row["square_id"]!r}'
        square_ids.add(sid)

    expected_ids = set(range(n))
    if square_ids != expected_ids:
        return f'n={n}: square_ids should be 0..{n - 1}, got {sorted(square_ids)}'

    return None


def _build_squares(n: int, rows: list[dict]) -> tuple[list[Square], mpmath.mpf]:
    rows_by_id = {int(r['square_id']): r for r in rows}

    raw_coords = []
    for sid in range(n):
        row = rows_by_id[sid]
        x_str = row['x'].strip()
        y_str = row['y'].strip()
        angle_str = row['angle'].strip()

        try:
            x_f = parse_value_float(x_str)
            y_f = parse_value_float(y_str)
            angle_f = parse_value_float(angle_str)
        except ValueError as e:
            raise ParticipantVisibleError(f'n={n}, square {sid}: cannot parse coordinates: {e}')

        if abs(x_f) > _MAX_COORD or abs(y_f) > _MAX_COORD:
            raise ParticipantVisibleError(f'n={n}, square {sid}: coordinate out of bounds (|x|, |y| must be <= {_MAX_COORD:g})')
        if abs(angle_f) > _MAX_ANGLE:
            raise ParticipantVisibleError(f'n={n}, square {sid}: angle out of bounds (|angle| must be <= {_MAX_ANGLE:g} radians)')

        raw_coords.append((sid, x_str, y_str, angle_str))

    x0_mp = parse_value_mp(raw_coords[0][1], L_DPS)
    y0_mp = parse_value_mp(raw_coords[0][2], L_DPS)
    two_pi = 2 * mpmath.pi

    squares = []
    for sid, x_str, y_str, angle_str in raw_coords:
        x_mp = parse_value_mp(x_str, L_DPS)
        y_mp = parse_value_mp(y_str, L_DPS)
        angle_mp = parse_value_mp(angle_str, L_DPS)

        dx_f = float(x_mp - x0_mp)
        dy_f = float(y_mp - y0_mp)
        angle_reduced_f = float(angle_mp % two_pi)

        squares.append(
            Square(
                x=dx_f,
                y=dy_f,
                angle=angle_reduced_f,
                x_str=x_str,
                y_str=y_str,
                angle_str=angle_str,
            )
        )

    return squares, measure_side_mp(squares)


def evaluate_single_puzzle(n: int, rows: list[dict]) -> tuple[int, float, str | None, str]:
    """Evaluate one puzzle, returning ``(n, score, L, reason)``.

    ``L`` is decimal text with ``L_DECIMALS`` digits after the point, and is
    reported only for a packing that passed the feasibility check: an
    unparseable, malformed, or overlapping puzzle gets None, so no L is
    published for a packing the metric rejected.  Text rather than a float
    because a double cannot hold those digits; the score itself is a double,
    taken from the same high-precision measurement.
    """
    error = _validate_puzzle(n, rows)
    if error:
        return (n, infeasible_score(n), None, error)

    try:
        squares, s_mp = _build_squares(n, rows)
    except ParticipantVisibleError as e:
        return (n, infeasible_score(n), None, str(e))

    feasible, reason = verify_puzzle(squares)
    if not feasible:
        return (n, infeasible_score(n), None, f'infeasible: {reason}')

    L_B = math.sqrt(n)
    if float(s_mp) < L_B - 1e-7:
        return (
            n,
            infeasible_score(n),
            None,
            f'infeasible: L={float(s_mp):.6f} is below area lower bound sqrt({n})={L_B:.6f}',
        )

    puzzle_score = max(0.0, score_puzzle(n, float(s_mp)))
    return (n, puzzle_score, _format_detail(s_mp), 'ok')


def _evaluate_puzzle_worker(args: tuple) -> tuple[int, float, str | None, str]:
    n, rows = args
    try:
        return evaluate_single_puzzle(n, rows)
    except Exception as e:  # noqa: BLE001 - one bad puzzle must not kill the pool
        return (n, infeasible_score(n), None, f'worker error: {e}')


def _evaluate_puzzles_parallel(
    puzzles: dict[int, list[dict]],
    puzzle_ns: set[int],
) -> tuple[float, list[tuple[int, float, str | None, str]]]:
    """Score every n in ``puzzle_ns``; rows for any other n are ignored."""
    work = [(n, rows) for n, rows in sorted(puzzles.items()) if n in puzzle_ns]

    if not work:
        results: list[tuple[int, float, str | None, str]] = []
    elif len(work) < MIN_PUZZLES_FOR_MP:
        results = [evaluate_single_puzzle(n, rows) for n, rows in work]
    else:
        with multiprocessing.Pool(processes=KAGGLE_CORES) as pool:
            results = pool.map(_evaluate_puzzle_worker, work)

    # Any scored n missing from the submission gets the grid-ceiling penalty
    # (lower-is-better metric — otherwise omitting a puzzle would improve the
    # total).
    seen = {n for n, _, _, _ in results}
    for n in puzzle_ns:
        if n not in seen:
            results.append((n, infeasible_score(n), None, 'missing from submission'))
    results.sort(key=lambda r: r[0])

    total = sum(puzzle_score for _, puzzle_score, _, _ in results)
    return total, results


def _format_detail(value: mpmath.mpf) -> str:
    """Decimal text for one per-puzzle L: exactly ``L_DECIMALS`` digits after the point.

    An ``mpf`` is a binary fraction (``man * 2**exp``), so it is converted
    exactly to a ``Fraction`` and rounded once, half-to-even, at the last
    written digit.  ``mpmath.nstr`` counts significant digits rather than
    decimal places, and ``format(mpf, '.50f')`` needs mpmath >= 1.4, so neither
    is usable here.

    >>> _format_detail(mpmath.mpf(2))[:12]
    '2.0000000000'
    >>> len(_format_detail(mpmath.mpf(2)).split('.')[1])
    50
    >>> _format_detail(parse_value_mp('1.41421356237309504880168872420969807856967187537695'))
    '1.41421356237309504880168872420969807856967187537695'
    """
    exact = Fraction(value.man) * Fraction(2) ** value.exp
    scale = 10**L_DECIMALS
    units = round(exact * scale)
    sign = '-' if units < 0 else ''
    whole, frac = divmod(abs(units), scale)
    return f'{sign}{whole}.{frac:0{L_DECIMALS}d}'


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
) -> float:
    """Kaggle entry point. Score a submission against the solution's puzzle set.

    ``solution`` is read for one thing only: the set of n to score.  Everything
    else about a puzzle -- best-known side length, proven-optimal flag, source
    -- is irrelevant to ``L / sqrt(n) - 1``.

    >>> import pandas as pd
    >>> sol = pd.DataFrame({"id": [0], "n": [1]})
    >>> sub = pd.DataFrame({"id": [0], "n": [1], "square_id": [0],
    ...     "x": ["0.5"], "y": ["0.5"], "angle": ["0"]})
    >>> score(sol, sub, "id")  # single unit square, L=1, L/sqrt(1) - 1 = 0.0
    0.0
    """
    solution = solution.drop(columns=[row_id_column_name])
    submission = submission.drop(columns=[row_id_column_name])

    puzzle_ns = {int(n) for n in solution['n']}

    text = submission.to_csv(index=False)
    puzzles = _parse_submission_csv(text)
    total, results = _evaluate_puzzles_parallel(puzzles, puzzle_ns)

    # L is the measured bounding side for puzzle n -- NOT the per-puzzle score
    # L/sqrt(n) - 1.  Puzzles with no accepted L (missing from the submission,
    # unparseable, failing column validation, or rejected as infeasible) keep
    # the historical 0 placeholder: an L measured off an overlapping packing is
    # not a real packing of n squares, and downstream harvesting treats every
    # published L as attainable.  Values arrive here already rendered as decimal
    # text with L_DECIMALS digits after the point; unlike a JSON float they are
    # not re-serialized downstream, so all 50 digits reach the stored file
    # verbatim.
    result_map = {n: L_text for n, _, L_text, _ in results if L_text is not None}
    zero = _format_detail(mpmath.mpf(0))

    output_dir = Path('/kaggle/working')
    if not output_dir.is_dir():
        output_dir = Path('.')
    with open(output_dir / 'submission_details.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['n', 'L'])
        for n in range(1, MAX_N + 1):
            writer.writerow([n, result_map.get(n, zero)])

    return float(total)

In [ ]:
# ── Import the metric and check it is the copy we expect ─────────────────────────────────────
import importlib
_body = Path("sis_metric.py").read_text().split("\n", 1)[1].rstrip("\n")   # drop the provenance comment line
assert hashlib.sha256(_body.encode()).hexdigest() == "d269f24b02d5f33fbd2149f76b83414b3b446be4d02776560e04631af3365ccf", "metric copy changed"
sys.path.insert(0, str(ROOT))
import sis_metric as metric
importlib.reload(metric)
log("metric loaded, sha256 d269f24b02d5f33f")

## The contact LP

A packing is tight when no small motion of the squares can shorten the bounding side without opening an overlap.
The LP checks exactly that, to first order:

- variables: a shift `(dx, dy)` and a turn `da` for every square, the four box edges and the side `L`, all inside a trust region `|·| ≤ r`;
- for every pair of squares within `4r` of contact, the separating-axis gap along the axis that currently separates them is
  linearised and kept from shrinking (`≥ 1e-12`, or its current value for pairs already in contact);
  aligned squares sit on a kink of the gap, so both one-sided slopes are added as two constraints;
- every corner within `4r` of the box is kept inside the box; the objective is `L`.

The step is applied with exact rotations and accepted only if the true side drops and no pair gets closer than float noise;
otherwise `r` shrinks. A packing that fails the metric only after rounding is pushed out by `1 + δ` as in the merge notebook.
With a penalty weight the same LP also *repairs* overlaps (slack on each violated pair), which is what the second experiment uses.

In [ ]:
# ── Geometry, the contact LP and certification ────────────────────────────────────────────────
# Idea: contact-LP polish (sigmaborov, Certified Contact Polish; octaviograu) and squeeze (malyshevdanil).
from scipy.optimize import linprog
from scipy.spatial import cKDTree
from scipy import sparse

SU = np.array([1.0, 1.0, -1.0, -1.0])
SV = np.array([1.0, -1.0, -1.0, 1.0])
HALF_DIAG = math.sqrt(2.0) / 2.0


def corners(P):
    """Corner coordinates (n, 4) of squares P = [[x, y, angle], ...]."""
    x, y, a = P[:, 0], P[:, 1], P[:, 2]
    c, s = np.cos(a)[:, None], np.sin(a)[:, None]
    cx = x[:, None] + 0.5 * (SU * c - SV * s)
    cy = y[:, None] + 0.5 * (SU * s + SV * c)
    return cx, cy


def side(P):
    """Bounding side L, measured the way the metric measures it."""
    cx, cy = corners(P)
    return max(cx.max() - cx.min(), cy.max() - cy.min())


def _h(psi):
    """Half-width of a unit square rotated by psi, projected on an axis."""
    return 0.5 * (np.abs(np.cos(psi)) + np.abs(np.sin(psi)))


def _dh(psi):
    """d h / d psi (one-sided value is chosen by the caller at kinks)."""
    return 0.5 * (-np.sign(np.cos(psi)) * np.sin(psi) + np.sign(np.sin(psi)) * np.cos(psi))


def close_pairs(P, cutoff):
    """Index pairs whose centres are close enough to come within `cutoff` of contact."""
    tree = cKDTree(P[:, :2])
    pairs = tree.query_pairs(2 * HALF_DIAG + cutoff, output_type="ndarray")
    return pairs[:, 0], pairs[:, 1]


def pair_gap(P, I, J):
    """Exact SAT gap of square pairs (>0 apart, <0 overlap) and the axis that realises it."""
    x, y, a = P[:, 0], P[:, 1], P[:, 2]
    dx, dy = x[J] - x[I], y[J] - y[I]
    phis = np.stack([a[I], a[I] + np.pi / 2, a[J], a[J] + np.pi / 2], 1)
    d = dx[:, None] * np.cos(phis) + dy[:, None] * np.sin(phis)
    gap = np.abs(d) - _h(a[I][:, None] - phis) - _h(a[J][:, None] - phis)
    k = gap.argmax(1)
    r = np.arange(len(I))
    return gap[r, k], k, phis[r, k], np.where(d[r, k] >= 0, 1.0, -1.0)


def lp_step(P, r, margin, weight=None):
    """One trust-region LP: minimise L (+ weight * overlap slack) with contacts linearised.

    Returns the proposed P (angles moved exactly; corners are recomputed from them).
    """
    n = len(P)
    x, y, a = P[:, 0], P[:, 1], P[:, 2]
    cut = 4 * r + 1e-12
    I, J = close_pairs(P, cut)
    gap0, k, phi, sg = pair_gap(P, I, J)
    keep = gap0 < cut
    I, J, gap0, k, phi, sg = I[keep], J[keep], gap0[keep], k[keep], phi[keep], sg[keep]
    own_i = k < 2                                          # axis belongs to square i (else to j)
    O = np.where(own_i, I, J)
    Q = np.where(own_i, J, I)
    dX, dY = x[J] - x[I], y[J] - y[I]
    cphi, sphi = np.cos(phi), np.sin(phi)
    psi = a[Q] - phi
    dh = _dh(psi)
    kink = (np.abs(np.sin(psi)) < 1e-7) | (np.abs(np.cos(psi)) < 1e-7)

    nv = 3 * n + 5                                         # dx, dy, da per square, X0, X1, Y0, Y1, L
    iX0, iX1, iY0, iY1, iL = 3 * n, 3 * n + 1, 3 * n + 2, 3 * n + 3, 3 * n + 4
    rows, cols, vals, rhs = [], [], [], []
    slack_rows = []

    def add_pair_rows(sel, dhv):
        m = int(sel.sum())
        if m == 0:
            return
        ii, jj, oo, qq = I[sel], J[sel], O[sel], Q[sel]
        s_, c_, sn_ = sg[sel], cphi[sel], sphi[sel]
        g_xj, g_yj = s_ * c_, s_ * sn_
        g_ao = s_ * (-dX[sel] * sn_ + dY[sel] * c_) + dhv
        g_aq = -dhv
        target = np.where(gap0[sel] >= margin, margin, np.minimum(gap0[sel], margin))
        if weight is not None:
            target = np.where(gap0[sel] < 0, margin, target)
        base = len(rhs)
        rid = np.arange(base, base + m)
        # -grad . dz <= gap0 - target
        for col, val in ((jj, -g_xj), (n + jj, -g_yj), (ii, g_xj), (n + ii, g_yj),
                         (2 * n + oo, -g_ao), (2 * n + qq, -g_aq)):
            rows.append(rid); cols.append(col); vals.append(val)
        rhs.extend((gap0[sel] - target).tolist())
        if weight is not None:
            slack_rows.extend(rid[gap0[sel] < margin].tolist())

    add_pair_rows(~kink, dh[~kink])
    add_pair_rows(kink, np.full(int(kink.sum()), 0.5))
    add_pair_rows(kink, np.full(int(kink.sum()), -0.5))

    cx, cy = corners(P)
    for cc, oth, cen, lo, hi, dcol in ((cx, cy, y, iX0, iX1, 0), (cy, cx, x, iY0, iY1, n)):
        sq, cn = np.nonzero(cc <= cc.min() + cut)
        # X0 <= c0 + d + dc/da * da   ->  X0 - d - dcda*da <= c0
        dcda = -(oth[sq, cn] - cen[sq]) if dcol == 0 else (oth[sq, cn] - cen[sq])
        m = len(sq); rid = np.arange(len(rhs), len(rhs) + m)
        rows += [rid, rid, rid]; cols += [np.full(m, lo), dcol + sq, 2 * n + sq]
        vals += [np.ones(m), -np.ones(m), -dcda]; rhs.extend(cc[sq, cn].tolist())
        sq, cn = np.nonzero(cc >= cc.max() - cut)
        dcda = -(oth[sq, cn] - cen[sq]) if dcol == 0 else (oth[sq, cn] - cen[sq])
        m = len(sq); rid = np.arange(len(rhs), len(rhs) + m)
        rows += [rid, rid, rid]; cols += [np.full(m, hi), dcol + sq, 2 * n + sq]
        vals += [-np.ones(m), np.ones(m), dcda]; rhs.extend((-cc[sq, cn]).tolist())
    for lo, hi in ((iX0, iX1), (iY0, iY1)):              # X1 - X0 - L <= 0
        rid = np.array([len(rhs)])
        rows += [rid, rid, rid]; cols += [np.array([hi]), np.array([lo]), np.array([iL])]
        vals += [np.ones(1), -np.ones(1), -np.ones(1)]; rhs.append(0.0)

    ns = len(slack_rows)
    if ns:
        rows.append(np.array(slack_rows)); cols.append(np.arange(nv, nv + ns)); vals.append(-np.ones(ns))
    A = sparse.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
                          shape=(len(rhs), nv + ns))
    c = np.zeros(nv + ns); c[iL] = 1.0
    if ns:
        c[nv:] = weight
    bounds = [(-r, r)] * (3 * n) + [(None, None)] * 5 + [(0, None)] * ns
    res = linprog(c, A_ub=A, b_ub=np.array(rhs), bounds=bounds, method="highs")
    if res.status != 0:
        return None
    dz = res.x
    Pn = P.copy()
    Pn[:, 0] += dz[:n]; Pn[:, 1] += dz[n:2 * n]; Pn[:, 2] += dz[2 * n:3 * n]
    return Pn


def violation(P):
    """Total overlap depth (0 for a valid packing up to float noise)."""
    I, J = close_pairs(P, 1e-9)
    g = pair_gap(P, I, J)[0]
    return float(np.clip(-g, 0, None).sum()), (float(g.min()) if len(g) else 1.0)


def squeeze(P, deadline, r0=1e-3, r_min=1e-10, iters=80, repair_weight=None):
    """Trust-region LP loop: shrink L while no contact gets worse than float noise.

    With repair_weight the loop first removes overlaps (merit = L + w * overlap) and then squeezes.
    """
    P = P.copy()
    L = side(P)
    viol, _ = violation(P)
    r = r0
    for _ in range(iters):
        if r < r_min or time.time() > deadline:
            break
        w = repair_weight if (repair_weight is not None and viol > CFG["TOL"]) else None
        Pn = lp_step(P, r, CFG["MARGIN"], w)
        if Pn is None:
            r *= 0.25
            continue
        Ln = side(Pn)
        vn, gmin = violation(Pn)
        if w is not None:
            ok = Ln + w * vn < L + w * viol - 1e-15
        else:
            ok = (gmin >= -CFG["TOL"]) and (Ln < L - 1e-15)
        if ok:
            P, L, viol = Pn, Ln, vn
            r = min(r * 2, r0)
        else:
            r *= 0.25
    return P, L, viol


def fmt_rows(n, P):
    """Rows for the metric: the doubles written as their shortest round-trip text."""
    return [{"n": str(n), "square_id": str(i), "x": repr(float(p[0])), "y": repr(float(p[1])),
             "angle": repr(float(p[2]))} for i, p in enumerate(P)]


def certify(n, rows):
    """The metric's verdict on one puzzle."""
    _, score, L_text, reason = metric.evaluate_single_puzzle(n, rows)
    return reason == "ok", score, reason


def certify_or_rescue(n, P):
    """Certify P; if only a near contact fails after rounding, push centres out by 1 + delta (as in 01)."""
    rows = fmt_rows(n, P)
    ok, score, reason = certify(n, rows)
    if ok:
        return rows, score
    if "float check" in reason:
        return None, None
    cx, cy = corners(P)
    mid = np.array([(cx.min() + cx.max()) / 2, (cy.min() + cy.max()) / 2])
    for delta in CFG["RESCUE_DELTAS"]:
        Q = P.copy()
        Q[:, :2] = mid + (P[:, :2] - mid) * (1 + delta)
        rows = fmt_rows(n, Q)
        ok, score, _ = certify(n, rows)
        if ok:
            return rows, score
    return None, None


log("geometry and contact LP ready")

## Baseline

The starting point is the certified submission of our merge notebook
[Every Public Packing, Certified](https://www.kaggle.com/code/flexonafft/every-public-packing-certified) (LB 22.257), attached as an input,
with its `per_puzzle.csv` for the exact per-puzzle scores. Puzzles that are not improved are written back byte for byte.

In [ ]:
# ── Baseline: the certified submission of 01 · Every Public Packing, Certified ────────────────
def find_input(name, prefer="every-public-packing"):
    """Path of an attached file, preferring the output of our 01 notebook."""
    found = sorted(INPUT.rglob(name))
    pref = [p for p in found if prefer in str(p)]
    return (pref or found or [None])[0]


_sub_path = find_input("submission.csv")
assert _sub_path is not None, "attach the output of 'Every Public Packing, Certified' as an input"
_base = pd.read_csv(_sub_path, dtype=str)
BASE_ROWS, BASE_P = {}, {}
for _n, _g in _base.groupby(_base["n"].astype(int)):
    _g = _g.assign(_sid=_g["square_id"].astype(int)).sort_values("_sid")
    BASE_ROWS[_n] = _g[COLS].to_dict("records")                # exact text, written back unchanged
    BASE_P[_n] = _g[["x", "y", "angle"]].astype(float).to_numpy()

_pp_path = find_input("per_puzzle.csv")
if _pp_path is not None:
    _pp = pd.read_csv(_pp_path)
    BASE_SCORE = dict(zip(_pp["n"].astype(int), _pp["score"].astype(float)))
else:                                                          # float estimate; the final metric pass is exact
    BASE_SCORE = {n: side(P) / math.sqrt(n) - 1 for n, P in BASE_P.items()}
for _n in range(1, 1001):
    BASE_SCORE.setdefault(_n, CEIL[_n])
BASE_TOTAL = sum(BASE_SCORE[n] for n in range(1, 1001))
BEATS_GRID = [n for n in range(1, 1001) if n in BASE_P and BASE_SCORE[n] < CEIL[n] - 1e-12]
log(f"baseline: {_sub_path.parent.name}/{_sub_path.name}, {len(BASE_P)} puzzles, total {BASE_TOTAL:.10f}, "
    f"{len(BEATS_GRID)} beat the grid")

## Kick, repair, squeeze

Each target is squeezed first, then hops until its time is up. A kick either shrinks the packing towards its centre by
`10^U(-6, -3.5)` or adds noise of `10^U(-4, -2)` to the positions and angles of the squares in one disc; the LP then removes the
overlaps (penalty mode) and squeezes. The best valid packing is certified by the metric at the end.

In [ ]:
# ── Kick and repair: hop to a nearby basin, then squeeze ──────────────────────────────────────
# Idea: basin hopping (alexchilton, Fault-line DP + Basin Hopping); shrink-and-repair (malyshevdanil, Neighbour Transfer + Squeeze).
import multiprocessing as mp

RUN_DEADLINE = NB_T0 + CFG["BUDGET_S"]


def kick(P, rng):
    """Either shrink the whole packing a little or jiggle one local patch; both create overlaps to repair."""
    Q = P.copy()
    cx, cy = corners(P)
    mid = np.array([(cx.min() + cx.max()) / 2, (cy.min() + cy.max()) / 2])
    if rng.random() < CFG["P_SHRINK"]:
        eps = 10 ** rng.uniform(*CFG["SHRINK_LOG10"])
        Q[:, :2] = mid + (P[:, :2] - mid) * (1 - eps)
        return Q, f"shrink {eps:.1e}"
    centre = P[rng.integers(len(P)), :2]
    rad = rng.uniform(*CFG["PATCH_RADIUS"])
    sel = np.hypot(*(P[:, :2] - centre).T) < rad
    sig = 10 ** rng.uniform(*CFG["JIGGLE_LOG10"])
    Q[sel, :2] += rng.normal(0, sig, (sel.sum(), 2))
    Q[sel, 2] += rng.normal(0, sig, sel.sum())
    return Q, f"jiggle {sel.sum()} sq, {sig:.1e}"


def work(n):
    """Squeeze, then kick-and-repair until the per-puzzle budget runs out; keep the shortest valid packing."""
    t0 = time.time()
    if t0 > RUN_DEADLINE:
        return n, None, None, 0.0, "budget", 0
    deadline = min(RUN_DEADLINE, t0 + CFG["PER_PUZZLE_S"])
    rng = np.random.default_rng(CFG["SEED"] + n)
    try:
        best, best_L, _ = squeeze(BASE_P[n], deadline, r0=CFG["R0"], iters=CFG["ITERS"])
        hops = 0
        while time.time() < deadline:
            Q, _ = kick(best, rng)
            Q, L, viol = squeeze(Q, deadline, r0=CFG["R0"], iters=CFG["ITERS"], repair_weight=CFG["REPAIR_W"])
            hops += 1
            if viol <= CFG["TOL"] and violation(Q)[1] >= -CFG["TOL"] and L < best_L - 1e-13:
                best, best_L = Q, L
    except Exception as e:
        return n, None, None, time.time() - t0, f"error: {e!r}"[:120], 0
    if best_L / math.sqrt(n) - 1 > BASE_SCORE[n] - CFG["MIN_GAIN"]:
        return n, None, None, time.time() - t0, "no gain", hops
    rows, score = certify_or_rescue(n, best)
    if rows is None or score > BASE_SCORE[n] - CFG["MIN_GAIN"]:
        return n, None, None, time.time() - t0, "not certified" if rows is None else "no gain", hops
    return n, rows, score, time.time() - t0, "improved", hops


TARGETS = [n for n in BEATS_GRID if CFG["N_RANGE"][0] <= n <= CFG["N_RANGE"][1]]
NEW, STATUS, HOPS = {}, Counter(), 0
_t0 = time.time()
with mp.Pool(CFG["WORKERS"]) as _pool:
    for _i, (_n, _rows, _score, _dt, _st, _h) in enumerate(_pool.imap_unordered(work, TARGETS, chunksize=1)):
        STATUS[_st.split(":")[0]] += 1
        HOPS += _h
        if _rows is not None:
            NEW[_n] = (_rows, _score)
            log(f"  n={_n:4d}  {BASE_SCORE[_n]:.12f} -> {_score:.12f}  gain {BASE_SCORE[_n]-_score:.3e}  ({_h} hops)")
        if (_i + 1) % 25 == 0:
            log(f"  {_i+1}/{len(TARGETS)} done, {len(NEW)} improved, gain so far "
                f"{sum(BASE_SCORE[k]-v[1] for k, v in NEW.items()):.3e}")
log(f"kick-and-repair: {len(TARGETS)} targets, {HOPS} hops, {dict(STATUS)}, {len(NEW)} improved, "
    f"gain {sum(BASE_SCORE[k]-v[1] for k, v in NEW.items()):.6e}, {time.time()-_t0:.0f}s")

## Write and score

Improved puzzles replace the baseline, everything else is copied as text. The file is read back the way Kaggle reads it and scored by
the metric end to end; with `per_puzzle.csv` attached the run stops if that total disagrees with the per-puzzle sums.

In [ ]:
# ── Write the merged submission and score it end to end ───────────────────────────────────────
_t0 = time.time()
_submission = ROOT / "submission.csv"
_rid = 0
with _submission.open("w", newline="") as _f:
    _w = csv.writer(_f)
    _w.writerow(["id", *COLS])
    for _n in sorted(BASE_ROWS):
        for _r in (NEW[_n][0] if _n in NEW else BASE_ROWS[_n]):
            _w.writerow([_rid, *(_r[c] for c in COLS)])
            _rid += 1

_sub = pd.read_csv(_submission, float_precision="round_trip")
_total = metric.score(pd.DataFrame({"id": range(1000), "n": range(1, 1001)}), _sub, "id")
_expected = sum(NEW[n][1] if n in NEW else BASE_SCORE[n] for n in range(1, 1001))
if _pp_path is not None:
    assert abs(_total - _expected) < 1e-9, (_total, _expected)

IMPROVED = pd.DataFrame([{"n": n, "base": BASE_SCORE[n], "new": NEW[n][1], "gain": BASE_SCORE[n] - NEW[n][1]}
                         for n in sorted(NEW)], columns=["n", "base", "new", "gain"])
IMPROVED.to_csv(ROOT / "improved.csv", index=False)
log(f"write: {_rid} rows, total {_total:.12f} (baseline {BASE_TOTAL:.12f}, gain {BASE_TOTAL-_total:.3e}), "
    f"{len(NEW)} puzzles replaced, {time.time()-_t0:.0f}s")

## Where the gain landed

In [ ]:
# ── Where the gain landed ─────────────────────────────────────────────────────────────────────
if len(IMPROVED):
    _k = np.floor(np.sqrt(IMPROVED["n"])); _u = (IMPROVED["n"] - _k**2) / (2 * _k + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))
    ax1.scatter(IMPROVED["n"], IMPROVED["gain"], s=18, color=ACCENT)
    ax1.set_yscale("log"); ax1.set_xlabel("n"); ax1.set_ylabel("score gain")
    ax1.set_title(f"{len(IMPROVED)} puzzles improved, total {IMPROVED['gain'].sum():.3e}")
    ax2.scatter(_u, IMPROVED["gain"], s=18, color=SECOND)
    ax2.set_yscale("log"); ax2.set_xlabel("band position u = (n - k²)/(2k + 1)"); ax2.set_ylabel("score gain")
    ax2.set_title("gain by position inside the band")
    plt.tight_layout(); plt.show()
    display(IMPROVED.sort_values("gain", ascending=False).head(20).style.format({"base": "{:.10f}", "new": "{:.10f}", "gain": "{:.3e}"}))
else:
    display(Markdown("**No puzzle improved.** The public packings are already tight for this move set; see the notes below."))
log(f"done in {time.time() - NB_T0:.0f}s")

## Notes

- The kicks are deliberately small: they look for a better packing *next to* the public one, not for a new structure.
- If the gain concentrates at a few positions in the band, the next step is to spend all hops there and widen the kicks.
- Per-puzzle results are in `improved.csv`; please credit the source of the starting packing listed in the merge notebook's `per_puzzle.csv`.